# Block 03 – Memory, one turn per cell

**You'll see:** the `messages` list grow turn by turn, a new thread starting blank, and the checkpoints behind it.

Run each turn cell **once**. Re-running a cell adds another turn to the same thread.

In [ ]:
# Setup: run this cell first. It finds the repo, makes it importable and loads .env
import os, sys, pathlib

def find_repo_root():
    starts = []
    if "__vsc_ipynb_file__" in globals():                 # VS Code tells us where this notebook is
        starts.append(pathlib.Path(globals()["__vsc_ipynb_file__"]).resolve().parent)
    starts.append(pathlib.Path.cwd().resolve())
    for start in starts:
        for folder in [start, *start.parents]:
            if (folder / "common" / "config.py").exists():
                return folder
            if (folder / "agents-weekend" / "common" / "config.py").exists():
                return folder / "agents-weekend"
    raise RuntimeError(f"Can't find the agents-weekend repo from {starts}. "
                       "Open the agents-weekend folder in VS Code, or set ROOT by hand.")

ROOT = find_repo_root()
os.chdir(ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
print("repo:", ROOT)

from common.config import get_llm, get_cheap_llm, DEFAULT_MODEL, CHEAP_MODEL
from common.tracing import langfuse_config, trace_attributes, flush, tracing_enabled
print("main model:", DEFAULT_MODEL, "| cheap model:", CHEAP_MODEL, "| Langfuse:", tracing_enabled())

In [ ]:
def describe(m):
    """One readable line per message."""
    kind = type(m).__name__.replace("Message", "")
    if getattr(m, "tool_calls", None):
        return f"{kind:<6} asks for: " + ", ".join(f"{tc['name']}({tc['args']})" for tc in m.tool_calls)
    text = (m.text or "").replace("\n", " ")
    return f"{kind:<6} {text[:90]}{'...' if len(text) > 90 else ''}"

## 1. Build the agent with a checkpointer

In [ ]:
from langgraph.checkpoint.memory import InMemorySaver
from common.graphs import build_tool_agent
from common.helpdesk import ALL_TOOLS, SYSTEM_PROMPT

saver = InMemorySaver()
graph = build_tool_agent(get_llm(), ALL_TOOLS, SYSTEM_PROMPT, checkpointer=saver, name="notebook-memory")
THREAD = "alice-notebook-1"
config = langfuse_config(thread_id=THREAD, recursion_limit=12)

def turn(text):
    with trace_attributes(session_id=THREAD, user_id="alice", tags=["notebook-03"]):
        result = graph.invoke({"messages": [{"role": "user", "content": text}]}, config=config)
    for i, m in enumerate(result["messages"]):
        print(f"[{i:>2}]", describe(m))
    print(f"\n{len(result['messages'])} messages in thread '{THREAD}'")

## 2. Turn 1

In [ ]:
turn("Hi, I'm alice. My VPN keeps dropping.")

## 3. Turn 2: 'it' only makes sense with memory

In [ ]:
turn("Is there an outage for it?")

## 4. Turn 3: 'I' = alice, from message [0]

In [ ]:
turn("OK. And what laptop do I have on record?")

## 5. A different thread starts blank

In [ ]:
bob = graph.invoke({"messages": [{"role": "user", "content": "What laptop do I have?"}]},
                   config=langfuse_config(thread_id="bob-notebook-1"))
print(bob["messages"][-1].text)

## 6. The checkpoints behind alice's thread
One snapshot per graph step; `next` is where a resume would continue.

In [ ]:
history = list(graph.get_state_history({"configurable": {"thread_id": THREAD}}))
for cp in reversed(history):
    msgs = cp.values.get("messages", [])
    print(f"step {cp.metadata.get('step'):>3} | next: {', '.join(cp.next) or 'END':<10} | {len(msgs):>2} messages")
print("\nthreads in the saver:", list(saver.storage.keys()))

In [ ]:
flush()